> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# Real26: open a completed covariance fit, then request compute explicitly

Individually migrated from `mega_gcorr_VIGOR_2_11_5_runtime_fixed(4).ipynb`.
This entrypoint preserves **read-first inspection**, native G/E factors and prediction-free animal embeddings. Set `VIGOR_COVARIANCE_FIT` to a completed store to open it **before loading phenotypes or the kernel**; no new optimizer is started unless `VIGOR_RUN_NEW_FIT=1`.
Without a saved store, the default clean-kernel path computes the full frozen 7,616 × 26 workflow and saves a new candidate checkpoint. See [MIGRATION.md](MIGRATION.md) for the 92-cell source lineage and deferred analyses. The GRM is transductive r11.2.1, and upstream trait pre-regression/family independence are unvalidated.

In [ ]:
from pathlib import Path
import os, sys
# Optional server overrides: VIGOR_SOURCE, VIGOR_REAL26_FIXTURE, VIGOR_PLINK_PREFIX,
# VIGOR_GDEC, VIGOR_RSCRIPT, VIGOR_OUTPUT, VIGOR_THREADS (1..20).
SOURCE = Path(os.environ.get('VIGOR_SOURCE', Path.cwd().parents[1])).resolve()
if not (SOURCE/'vigor'/'__init__.py').is_file():
    candidates = [Path.cwd(), *Path.cwd().parents]
    SOURCE = next(p for p in candidates if (p/'vigor'/'__init__.py').is_file())
sys.path.insert(0, str(SOURCE))
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[name] = os.environ.get('VIGOR_THREADS', '2')
import numpy as np, pandas as pd
get_ipython().run_line_magic('matplotlib', 'inline')
import matplotlib.pyplot as plt
import vigor
from examples.real26_workflow import (configuration, load_fixture, run_prediction,
    run_comparators, run_extended, write_json)
assert Path(vigor.__file__).resolve() == SOURCE/'vigor'/'__init__.py'
print({'version':vigor.__version__, 'source':'selected candidate', 'threads':os.environ['OMP_NUM_THREADS']})


## Read the named completed checkpoint before any input/kernel load

In [ ]:
from vigor.phenome import (open_covariance_fit, save_covariance_fit, native_factor_loadings,
    animal_factor_embeddings, phenotype_digest, add_covariance_se)
config = configuration('runtime')
saved_name = os.environ.get('VIGOR_COVARIANCE_FIT')
prior_fit = open_covariance_fit(Path(saved_name)) if saved_name else None
RUN_NEW_FIT = os.environ.get('VIGOR_RUN_NEW_FIT', '0' if prior_fit is not None else '1') == '1'
RUN_EMBEDDINGS = os.environ.get('VIGOR_RUN_EMBEDDINGS', '1' if RUN_NEW_FIT else '0') == '1'
RUN_SE = os.environ.get('VIGOR_RUN_SE', '0') == '1'
if prior_fit is None and not RUN_NEW_FIT:
    raise ValueError('Supply a completed fit or explicitly enable a new fit')
print({'opened_saved_fit':prior_fit is not None, 'new_fit_requested':RUN_NEW_FIT,
       'input_kernel_loaded':False, 'SE_requested':RUN_SE})

## Compute only the requested stages; preserve the saved fit

In [ ]:
need_inputs = RUN_NEW_FIT or RUN_EMBEDDINGS or RUN_SE or config['extended']
data = load_fixture(config) if need_inputs else None
prediction = run_prediction(data, config) if RUN_NEW_FIT else None
if prediction is not None:
    fit = prediction['covariance_fit']
    checkpoint = config['output']/'covariance.zarr'
    save_covariance_fit(fit, data['ztrain'], checkpoint,
        parameters={'selection':prediction['prediction_sparse_cv']['chosen']})
    reopened = open_covariance_fit(checkpoint)
    np.testing.assert_allclose(reopened['G_prediction'], fit['G_prediction'])
    metrics = pd.read_csv(config['output']/'vigor_metrics.csv')
    display(metrics.groupby(['target','split']).predictive_r2.mean().to_frame())
else:
    reopened = prior_fit
    write_json(config['output']/'reuse.json', {'state':'COMPLETE','optimizer_started':False,
        'input_kernel_loaded':need_inputs,'phenotype_digest_present':bool(reopened.get('phenotype_digest'))})
print({'shape':list(reopened['G'].shape),'inference':'unshrunk','prediction':'separate profiles'})

## Saved native factors and G/E correlation; no re-decomposition

In [ ]:
G_loadings = native_factor_loadings(reopened,'G')
E_loadings = native_factor_loadings(reopened,'E')
display(G_loadings.iloc[:5,:5])
fig, axes = plt.subplots(1,2,figsize=(12,4))
for ax,c in zip(axes,['G','E']):
    C = np.asarray(reopened[c]); d = np.sqrt(np.maximum(np.diag(C),1e-12))
    im = ax.imshow(C/d[:,None]/d[None,:],vmin=-1,vmax=1,cmap='RdBu_r')
    ax.set_title(c+' inference correlation, 26 traits')
fig.colorbar(im,ax=axes.tolist(),shrink=.7)
fig.savefig(config['output']/'GE_correlations.png',dpi=140)
plt.show()

## Prediction-free joint animal factor posteriors
These use the animal's own observed phenotypes, retaining all native G/E factors. They are **not genetic BLUPs**. Unobserved animals retain prior mean zero/SD one. A saved phenotype digest must match before reusing the fit for this demonstration.

In [ ]:
if RUN_EMBEDDINGS:
    if reopened.get('phenotype_digest') != phenotype_digest(data['ztrain']):
        raise ValueError('Saved fit/input phenotype digest differs; recover exact original inputs')
    embeddings = animal_factor_embeddings(data['ztrain'],reopened,center=False,
                                         blas_threads=config['threads'])
    assert embeddings.sizes['sample'] == len(data['ztrain'])
    assert np.isfinite(embeddings.G_trait_projection).all()
    write_json(config['output']/'embeddings.json',{'state':'COMPLETE',
        'shape_G':list(embeddings.G_trait_projection.shape),
        'shape_E':list(embeddings.E_trait_projection.shape),
        'solver_pattern_counts':embeddings.attrs['solver_pattern_counts']})
else:
    write_json(config['output']/'embeddings.json',{'state':'SKIPPED','reason':'explicit compute disabled'})

## Optional SE recovery and bounded discovery
`VIGOR_RUN_SE=1` requests eight analytic/trace probes conditional on the saved rank/support, with exact input provenance; it does not measure support-selection uncertainty. `VIGOR_RUN_EXTENDED=1` requests the bounded 512-animal, 128-marker, 26-trait direct-score LOCO example, 24-marker fine mapping and covariance exploration. Whole-genome, ontology enrichment, PC–ANM–SEM chains and SHAP remain unrun.

In [ ]:
if RUN_SE:
    recovered = add_covariance_se(data['K'],data['ztrain'],reopened,probes=8,trace_probes=8,
                                 blas_threads=config['threads'],progress=False)
    save_covariance_fit(recovered,data['ztrain'],config['output']/'covariance_with_se.zarr')
    write_json(config['output']/'SE.json',{'state':'COMPLETE',
        'finite_G_se_entries':int(np.isfinite(recovered['G_se']).sum().sum()),
        'fixed_support':True,'probes':8,'trace_probes':8})
else:
    write_json(config['output']/'SE.json',{'state':'SKIPPED','reason':'VIGOR_RUN_SE=0'})
extended = run_extended(data,config, prediction or {'covariance_fit':reopened})
print(extended)

In [ ]:
write_json(config['output']/'notebook_summary.json',{'state':'COMPLETE',
    'mode':'new_fit' if RUN_NEW_FIT else 'saved_fit','new_fit_executed':RUN_NEW_FIT,
    'embeddings_executed':RUN_EMBEDDINGS,'SE_executed':RUN_SE,'extended_state':extended['state'],
    'unrun':['whole-genome/server scale','ontology enrichment without annotations','SHAP','PC-ANM-SEM chain','new A/D/GxE kernels']})
print('Completed requested stages; candidate approval remains pending.')